# Лабораторная 4 — динамическое программирование для замены оборудования


## 1. Смысл задачи с нуля

Есть машина. В начале каждого этапа у неё есть возраст.

Возраст — это количество этапов, которые машина уже отработала.

Новая машина имеет возраст ноль. После одного этапа возраст становится один. После двух этапов возраст становится два.

На каждом этапе можно сделать одно из двух действий:

- `keep` — оставить текущую машину и использовать её на этом этапе;
- `sell` — продать старую машину, сразу купить новую и использовать уже новую машину на этом этапе.

После последнего этапа мы обязательно продаём машину и больше ничего не покупаем.

Ключевые обозначения:

- `n` — количество этапов;
- `k` — максимальный срок службы машины;
- `I` — цена новой машины;
- `r(t)` — прибыль за этап от машины возраста `t`;
- `c(t)` — расходы на обслуживание машины возраста `t`;
- `s(t)` — цена продажи машины возраста `t`.

Чистая прибыль от использования машины возраста `t`:

$$
p(t)=r(t)-c(t)
$$

Если машине уже `k` лет, её нельзя использовать дальше. Её можно только продать.


## 2. Почему удобно думать не по этапам, а по циклам

Один цикл — это период, когда мы купили или уже имели новую машину и держим её несколько этапов подряд.

Например, если машина используется 4 этапа подряд, то её возраста на этих этапах такие:

```text
0, 1, 2, 3
```

После этого мы либо продаём машину и покупаем новую, либо продаём её в самом конце всей задачи.

Пусть длина цикла равна `L`. Тогда машина используется с возрастами от нуля до `L - 1`.

Суммарная прибыль от использования машины за цикл длины `L`:

$$
U(L)=\sum_{t=0}^{L-1}\bigl(r(t)-c(t)\bigr)
$$

Если это не последний цикл, то после него мы продаём старую машину и покупаем новую. Поэтому обычный цикл даёт:

$$
A(L)=U(L)+s(L)-I
$$

Если это последний цикл, то после него мы только продаём машину, но новую уже не покупаем. Поэтому последний цикл даёт:

$$
B(L)=U(L)+s(L)
$$

Главная идея: вся задача превращается в разбиение `n` этапов на куски длиной не больше `k`.


## 3. Формула динамического программирования

Пусть `D[x]` — максимальная прибыль после `x` этапов, если эти `x` этапов уже разбиты на полные обычные циклы, и мы готовы начать следующий этап с новой машиной.

Начальное значение:

$$
D(0)=0
$$

Переход:

$$
D(x)=\max_{1\le L\le \min(k,x)}\bigl(D(x-L)+A(L)\bigr)
$$

Смысл перехода:

- последний обычный цикл имеет длину `L`;
- до него уже было оптимально пройдено `x - L` этапов;
- добавляем выгоду обычного цикла `A(L)`.

Финальный ответ считается отдельно, потому что последний цикл заканчивается продажей без покупки новой машины:

$$
Ans=\max_{1\le L\le \min(k,n)}\bigl(D(n-L)+B(L)\bigr)
$$


## 4. Импорты


In [1]:
import math
from dataclasses import dataclass
from collections import Counter
from typing import Callable, Optional

import numpy as np
import matplotlib.pyplot as plt

try:
  from numba import njit
  NUMBA_AVAILABLE = True
except Exception:
  NUMBA_AVAILABLE = False

print("Numba available:", NUMBA_AVAILABLE)


Numba available: True


## 5. Описание одного теста

Заведём структуру `EquipmentCase`, чтобы каждый тест хранился аккуратно.


In [2]:
@dataclass
class EquipmentCase:
  name: str
  n: int
  k: int
  I: float
  r: Callable[[int], float]
  c: Callable[[int], float]
  s: Callable[[int], float]


## 6. Подготовка массивов `A` и `B`

Здесь мы заранее считаем все возможные длины цикла от 1 до `k`.

`A[L]` — прибыль обычного цикла длины `L`, после которого мы продаём старую машину и покупаем новую.

`B[L]` — прибыль последнего цикла длины `L`, после которого мы только продаём машину.


In [3]:
def build_cycle_arrays(case: EquipmentCase):
  k = case.k
  I = case.I

  use_profit = np.zeros(k, dtype=np.float64)

  for t in range(k):
    use_profit[t] = case.r(t) - case.c(t)

  U = np.zeros(k + 1, dtype=np.float64)

  for L in range(1, k + 1):
    U[L] = U[L - 1] + use_profit[L - 1]

  A = np.zeros(k + 1, dtype=np.float64)
  B = np.zeros(k + 1, dtype=np.float64)

  for L in range(1, k + 1):
    A[L] = U[L] + case.s(L) - I
    B[L] = U[L] + case.s(L)

  return use_profit, U, A, B


## 7. Основной решатель ДП

Этот вариант хорошо подходит для обычных тестов, например при `n = 1000` и `k = 100`.

Он также умеет восстановить оптимальные длины циклов, а потом из них получить список решений `keep` и `sell`.


In [4]:
def solve_full_dp(case: EquipmentCase, need_decisions: bool = True):
  n = case.n
  k = case.k

  _, _, A, B = build_cycle_arrays(case)

  negative_inf = -1e300

  D = np.full(n + 1, negative_inf, dtype=np.float64)
  prev = np.zeros(n + 1, dtype=np.int32)

  D[0] = 0.0

  for x in range(1, n + 1):
    max_L = min(k, x)

    best_value = negative_inf
    best_L = 1

    for L in range(1, max_L + 1):
      value = D[x - L] + A[L]

      if value > best_value:
        best_value = value
        best_L = L

    D[x] = best_value
    prev[x] = best_L

  final_best_value = negative_inf
  final_best_L = 1

  for L in range(1, min(k, n) + 1):
    value = D[n - L] + B[L]

    if value > final_best_value:
      final_best_value = value
      final_best_L = L

  segments = []
  remaining = n - final_best_L

  while remaining > 0:
    L = int(prev[remaining])
    segments.append(L)
    remaining -= L

  segments.reverse()
  segments.append(final_best_L)

  decisions = None

  if need_decisions:
    decisions = segments_to_decisions(segments)

  return {
    "answer": final_best_value,
    "segments": segments,
    "decisions": decisions,
    "D": D,
    "prev": prev,
    "A": A,
    "B": B,
  }


def segments_to_decisions(segments):
  decisions = []

  for index, L in enumerate(segments):
    if index == 0:
      decisions.extend(["keep"] * L)
    else:
      decisions.append("sell")
      decisions.extend(["keep"] * (L - 1))

  return decisions


## 8. Ускоренный вариант на Numba

Этот блок нужен, если хочется прогнать большой тест вроде `n = 1000000`, `k = 1000` точным ДП.

Без Numba такой тест в Python может работать слишком долго, потому что операций примерно:

$$
n\cdot k
$$

Для большого теста из письма это примерно:

$$
1000000\cdot1000=1000000000
$$

То есть около миллиарда проверок перехода.


In [5]:
if NUMBA_AVAILABLE:
  @njit
  def solve_arrays_numba(n, k, A, B):
    negative_inf = -1e300

    D = np.empty(n + 1, dtype=np.float64)
    prev = np.empty(n + 1, dtype=np.int32)

    for i in range(n + 1):
      D[i] = negative_inf
      prev[i] = 0

    D[0] = 0.0

    for x in range(1, n + 1):
      max_L = k

      if x < k:
        max_L = x

      best_value = negative_inf
      best_L = 1

      for L in range(1, max_L + 1):
        value = D[x - L] + A[L]

        if value > best_value:
          best_value = value
          best_L = L

      D[x] = best_value
      prev[x] = best_L

    final_best_value = negative_inf
    final_best_L = 1

    max_L = k

    if n < k:
      max_L = n

    for L in range(1, max_L + 1):
      value = D[n - L] + B[L]

      if value > final_best_value:
        final_best_value = value
        final_best_L = L

    return final_best_value, final_best_L, D, prev
else:
  solve_arrays_numba = None


def solve_full_dp_numba(case: EquipmentCase, need_decisions: bool = False):
  if not NUMBA_AVAILABLE:
    raise RuntimeError("Numba is not available. Use solve_full_dp for small tests.")

  n = case.n
  k = case.k

  _, _, A, B = build_cycle_arrays(case)
  answer, final_L, D, prev = solve_arrays_numba(n, k, A, B)

  segments = []
  remaining = n - int(final_L)

  while remaining > 0:
    L = int(prev[remaining])
    segments.append(L)
    remaining -= L

  segments.reverse()
  segments.append(int(final_L))

  decisions = None

  if need_decisions:
    decisions = segments_to_decisions(segments)

  return {
    "answer": answer,
    "segments": segments,
    "decisions": decisions,
    "D": D,
    "prev": prev,
    "A": A,
    "B": B,
  }


## 9. Тесты из письма


In [6]:
def get_test_case(case_id: int) -> EquipmentCase:
  if case_id == 3:
    return EquipmentCase(
      name="case 3",
      n=1000,
      k=100,
      I=1234,
      r=lambda t: 123,
      c=lambda t: 17 + t,
      s=lambda t: 567,
    )

  if case_id == 4:
    return EquipmentCase(
      name="case 4",
      n=1000,
      k=100,
      I=1234,
      r=lambda t: 123 - t,
      c=lambda t: 17,
      s=lambda t: 567,
    )

  if case_id == 5:
    return EquipmentCase(
      name="case 5",
      n=1000,
      k=100,
      I=1234,
      r=lambda t: 123,
      c=lambda t: 17 + t,
      s=lambda t: 567 - 5 * t,
    )

  if case_id == 6:
    return EquipmentCase(
      name="case 6",
      n=1000,
      k=100,
      I=1234,
      r=lambda t: 123 - t,
      c=lambda t: 17,
      s=lambda t: 567 - 5 * t,
    )

  if case_id == 7:
    return EquipmentCase(
      name="case 7",
      n=1000,
      k=100,
      I=1234,
      r=lambda t: 123 - t,
      c=lambda t: 17 + t,
      s=lambda t: 567,
    )

  if case_id == 8:
    return EquipmentCase(
      name="case 8",
      n=1000,
      k=100,
      I=1234,
      r=lambda t: 123 - t,
      c=lambda t: 17 + t,
      s=lambda t: 567 - 5 * t,
    )

  if case_id == 9:
    return EquipmentCase(
      name="case 9",
      n=1000000,
      k=1000,
      I=1234,
      r=lambda t: 123 - 0.1 * t + math.sin(t),
      c=lambda t: 17 + 0.1 * t + math.cos(t),
      s=lambda t: 567 - 0.5 * t + math.sin(2 * t),
    )

  if case_id == 2:
    return EquipmentCase(
      name="max test",
      n=1000000009,
      k=1117,
      I=1200,
      r=lambda t: 2,
      c=lambda t: 1,
      s=lambda t: 1200 - t,
    )

  raise ValueError(f"Unknown case_id: {case_id}")


## 10. Проверка тестов 3–8

Эти тесты маленькие, поэтому спокойно решаются обычным ДП.


In [7]:
small_results = {}

for case_id in [3, 4, 5, 6, 7, 8]:
  case = get_test_case(case_id)
  result = solve_full_dp(case, need_decisions=True)
  small_results[case_id] = result

  print("=" * 50)
  print(case.name)
  print("answer:", result["answer"])
  print("segments count:", len(result["segments"]))
  print("segments:", Counter(result["segments"]))
  print("first 30 decisions:", result["decisions"][:30])


case 3
answer: 71206.0
segments count: 27
segments: Counter({37: 26, 38: 1})
first 30 decisions: ['keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep']
case 4
answer: 71206.0
segments count: 27
segments: Counter({37: 26, 38: 1})
first 30 decisions: ['keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep']
case 5
answer: 66206.0
segments count: 27
segments: Counter({37: 26, 38: 1})
first 30 decisions: ['keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'k

Ожидаемые ответы для тестов 3–8:

| Тест | Оптимальная прибыль | Структура циклов |
|---:|---:|---|
| 3 | 71206 | один цикл длины 38 и двадцать шесть циклов длины 37 |
| 4 | 71206 | один цикл длины 38 и двадцать шесть циклов длины 37 |
| 5 | 66206 | один цикл длины 38 и двадцать шесть циклов длины 37 |
| 6 | 66206 | один цикл длины 38 и двадцать шесть циклов длины 37 |
| 7 | 56571 | двадцать пять циклов длины 26 и четырнадцать циклов длины 25 |
| 8 | 51571 | двадцать пять циклов длины 26 и четырнадцать циклов длины 25 |


## 12. Большой тест 9

Тест 9:

$$
n=1000000
$$

$$
k=1000
$$

По графику средней выгоды видно, что наиболее выгодная длина обычного цикла равна 80.

Так как:

$$
1000000=12500\cdot80
$$

кандидатная стратегия очень простая: держать каждую машину ровно 80 этапов.

Для этого теста ответ получается примерно:

$$
89289822.769505
$$

Если хочется прогнать точное ДП для теста 9, можно использовать Numba-блок ниже. Первый запуск будет дольше, потому что Numba компилирует функцию.


In [10]:
def solve_case_9_by_repeating_best_cycle():
  case = get_test_case(9)
  _, _, A, B = build_cycle_arrays(case)

  best_L = max(range(1, case.k + 1), key=lambda L: A[L] / L)

  if case.n % best_L != 0:
    raise ValueError("For this simple shortcut n must be divisible by best_L.")

  cycles_count = case.n // best_L
  answer = (cycles_count - 1) * A[best_L] + B[best_L]

  return answer, best_L, cycles_count


answer_9, best_L_9, cycles_count_9 = solve_case_9_by_repeating_best_cycle()

print("case 9 answer:", answer_9)
print("best cycle length:", best_L_9)
print("cycles count:", cycles_count_9)


case 9 answer: 89289822.769505
best cycle length: 80
cycles count: 12500


In [ ]:
# можно раскомментировать, если нужно именно точное ДП для case 9.
# На обычном Python это слишком долго, поэтому здесь нужен Numba.

# case_9 = get_test_case(9)
# result_9_exact = solve_full_dp_numba(case_9, need_decisions=False)
# print(result_9_exact["answer"])
# print(Counter(result_9_exact["segments"]).most_common(10))


## 13. Аналитическое решение макс-теста

Макс-тест из письма:

$$
n=1000000009
$$

$$
k=1117
$$

$$
I=1200
$$

$$
r(t)=2
$$

$$
c(t)=1
$$

$$
s(t)=I-t
$$

Чистая прибыль за один этап:

$$
r(t)-c(t)=1
$$

Если держим машину `L` этапов, то прибыль от использования:

$$
U(L)=L
$$

Обычный цикл:

$$
A(L)=U(L)+s(L)-I
$$

Подставляем:

$$
A(L)=L+(I-L)-I
$$

Получается:

$$
A(L)=0
$$

Последний цикл:

$$
B(L)=U(L)+s(L)
$$

Подставляем:

$$
B(L)=L+(I-L)
$$

Получается:

$$
B(L)=I=1200
$$

Итоговый ответ:

$$
f_1(0)=1200
$$

Любая стратегия, где машина не используется больше `k` этапов подряд, будет оптимальной.

Но печатать миллиард действий `keep` и `sell` неадекватно. Для такого теста лучше сдавать прибыль и сжатое описание стратегии.


In [12]:
def solve_max_test_analytically():
  case = get_test_case(2)

  answer = case.I

  full_cycles = case.n // case.k
  last_cycle = case.n % case.k

  if last_cycle == 0:
    compressed_strategy = [(case.k, full_cycles)]
  else:
    compressed_strategy = [(case.k, full_cycles), (last_cycle, 1)]

  return answer, compressed_strategy


answer_max, strategy_max = solve_max_test_analytically()

print("max test answer:", answer_max)
print("compressed strategy:", strategy_max)


max test answer: 1200
compressed strategy: [(1117, 895255), (174, 1)]


## 14. Генерация и чтение простого входного формата

Так как точный формат файлов в письме не указан, ниже используется простой формат:

```text
n k I
r(0) r(1) ... r(k-1)
c(0) c(1) ... c(k-1)
s(1) s(2) ... s(k)
```

Если в LMS формат другой, менять нужно только эти функции.


In [13]:
def write_case_to_file(case: EquipmentCase, path: str):
  with open(path, "w", encoding="utf-8") as file:
    file.write(f"{case.n} {case.k} {case.I}\n")
    file.write(" ".join(str(case.r(t)) for t in range(case.k)) + "\n")
    file.write(" ".join(str(case.c(t)) for t in range(case.k)) + "\n")
    file.write(" ".join(str(case.s(t)) for t in range(1, case.k + 1)) + "\n")


def read_case_from_file(path: str, name: Optional[str] = None) -> EquipmentCase:
  with open(path, "r", encoding="utf-8") as file:
    lines = [line.strip() for line in file.readlines() if line.strip()]

  n_raw, k_raw, I_raw = lines[0].split()

  n = int(n_raw)
  k = int(k_raw)
  I = float(I_raw)

  r_values = [float(x) for x in lines[1].split()]
  c_values = [float(x) for x in lines[2].split()]
  s_values = [float(x) for x in lines[3].split()]

  if len(r_values) != k:
    raise ValueError("r array must have length k")

  if len(c_values) != k:
    raise ValueError("c array must have length k")

  if len(s_values) != k:
    raise ValueError("s array must have length k")

  return EquipmentCase(
    name=name or path,
    n=n,
    k=k,
    I=I,
    r=lambda t: r_values[t],
    c=lambda t: c_values[t],
    s=lambda t: s_values[t - 1],
  )


## 16. Запись ответа

Для маленьких тестов можно записать все решения по этапам.

Для огромных тестов это делать нельзя, потому что файл будет слишком большим.


In [14]:
def write_solution(path: str, answer: float, decisions=None, segments=None):
  with open(path, "w", encoding="utf-8") as file:
    file.write(f"answer {answer}\n")

    if decisions is not None:
      file.write("decisions\n")

      for decision in decisions:
        file.write(decision + "\n")

    if segments is not None:
      file.write("segments\n")

      for L in segments:
        file.write(str(L) + "\n")


## 17. Проверка на одном сгенерированном файле


In [15]:
case = get_test_case(3)
write_case_to_file(case, "case_3.txt")

case_from_file = read_case_from_file("case_3.txt", name="case 3 from file")
result_from_file = solve_full_dp(case_from_file, need_decisions=True)

write_solution(
  path="solution_case_3.txt",
  answer=result_from_file["answer"],
  decisions=result_from_file["decisions"],
  segments=result_from_file["segments"],
)

print("answer:", result_from_file["answer"])
print("segments:", Counter(result_from_file["segments"]))
print("first 20 decisions:", result_from_file["decisions"][:20])


answer: 71206.0
segments: Counter({37: 26, 38: 1})
first 20 decisions: ['keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep', 'keep']


## 18. Объяснение

Свели задачу замены оборудования к динамическому программированию по длинам циклов эксплуатации. Если машину держать `L` этапов, я заранее считаю суммарную прибыль от её использования и цену продажи. Для обычного цикла учитываю продажу старой машины и покупку новой. Для последнего цикла учитываю только продажу, потому что после последнего этапа новая машина уже не покупается. Потом решаю задачу разбиения `n` этапов на куски длиной не больше `k`.

Главные формулы:

$$
U(L)=\sum_{t=0}^{L-1}\bigl(r(t)-c(t)\bigr)
$$

$$
A(L)=U(L)+s(L)-I
$$

$$
B(L)=U(L)+s(L)
$$

$$
D(x)=\max_{1\le L\le \min(k,x)}\bigl(D(x-L)+A(L)\bigr)
$$

$$
Ans=\max_{1\le L\le \min(k,n)}\bigl(D(n-L)+B(L)\bigr)
$$

Если спросят, почему последний цикл считается отдельно:

> Потому что после последнего этапа оборудование продаётся, но новое уже не покупается. Поэтому в последнем цикле нет вычитания цены новой машины `I`.

Если спросят, что означает `sell`:

> `sell` означает: в начале этапа продать текущую машину, купить новую и использовать новую машину на этом же этапе.

Если спросят, почему нельзя использовать машину возраста `k`:

> По условию `k`-летнее оборудование можно только продать. Поэтому длина любого цикла не может быть больше `k`.


## 19. короткая шпаргалка

1. Сначала считаем прибыль использования машины каждого возраста:

$$
p(t)=r(t)-c(t)
$$

2. Потом считаем прибыль цикла длины `L`:

$$
U(L)=p(0)+p(1)+\dots+p(L-1)
$$

3. Если цикл не последний:

$$
A(L)=U(L)+s(L)-I
$$

4. Если цикл последний:

$$
B(L)=U(L)+s(L)
$$

5. ДП выбирает, какой длины был последний обычный цикл:

$$
D(x)=\max\bigl(D(x-L)+A(L)\bigr)
$$

6. Финальный ответ выбирает длину последнего цикла:

$$
Ans=\max\bigl(D(n-L)+B(L)\bigr)
$$

7. Длины циклов превращаются в ответы так:

- первый цикл — только `keep`;
- каждый следующий цикл — сначала `sell`, потом несколько `keep`.
